In [2]:
import sys
from pathlib import Path
import os

print("Current working directory:", os.getcwd())

# If notebook is inside /notebooks
PROJECT_ROOT = Path().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

print("Project root added to sys.path:", PROJECT_ROOT)


Current working directory: /home/sreethanu/lidc_ldri_dataset_1/lung_cancer_vl_jepa_lidc/notebooks
Project root added to sys.path: /home/sreethanu/lidc_ldri_dataset_1


In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Does src exist?", (PROJECT_ROOT / "src").exists())


Project root: /home/sreethanu/lidc_ldri_dataset_1/lung_cancer_vl_jepa_lidc
Does src exist? True


In [37]:
import warnings
warnings.filterwarnings("ignore")
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, random_split, Dataset
from tqdm import tqdm
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path
from collections import defaultdict

from src.models.encoder_3d import ViT3DEncoder
from src.models.classification_head import ClassificationHead
from src.utils.config import Config
from src.utils.seed import set_global_seed
from src.utils.dataloader import LIDCPatchDataset
from src.utils.metrics import compute_classification_metrics


In [5]:
config_path = PROJECT_ROOT / "configs" / "jepa_config.yaml"
config = Config(str(config_path))

set_global_seed(
    config.get("project.seed"),
    deterministic=config.get("project.deterministic")
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Setting global seed: 42
Enabling deterministic mode (may reduce performance)
Reproducibility configured successfully.
Device: cuda


In [6]:
online_encoder = ViT3DEncoder(
    input_size=tuple(config.get("data.input_shape")[1:]),
    patch_size=tuple(config.get("model.encoder.patch_size")),
    embed_dim=config.get("model.encoder.embed_dim"),
    depth=config.get("model.encoder.depth"),
    num_heads=config.get("model.encoder.num_heads"),
).to(device)


In [7]:
checkpoint_path = PROJECT_ROOT / "checkpoints" / "pretraining" / "jepa_epoch_100.pth"

checkpoint = torch.load(checkpoint_path, map_location=device)

online_encoder.load_state_dict(checkpoint["online_encoder"])
online_encoder.eval()

print("Pretrained encoder loaded successfully.")


Pretrained encoder loaded successfully.


In [13]:


ANNOTATION_DIR = PROJECT_ROOT / "data" / "raw" / "LIDC-IDRI" / "ANNOTATIONS"
PATCH_DIR = PROJECT_ROOT / "data" / "processed" / "patches"

xml_files = list(ANNOTATION_DIR.rglob("*.xml"))
print("Total XML files found:", len(xml_files))

series_malignancy = defaultdict(list)

# Define namespace
ns = {"ns": "http://www.nih.gov"}

for xml_file in xml_files:
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        # Extract SeriesInstanceUid (note lowercase d)
        series_uid_elem = root.find(".//ns:SeriesInstanceUid", ns)
        if series_uid_elem is None:
            continue
        
        series_uid = series_uid_elem.text.strip()

        # Extract malignancy scores
        for malignancy_elem in root.findall(".//ns:malignancy", ns):
            try:
                score = int(malignancy_elem.text)
                series_malignancy[series_uid].append(score)
            except:
                continue

    except Exception:
        continue

print("Series with malignancy annotations:", len(series_malignancy))


Total XML files found: 1319
Series with malignancy annotations: 883


In [11]:
import xml.etree.ElementTree as ET
from pathlib import Path

xml_files = list(ANNOTATION_DIR.rglob("*.xml"))

print("Example XML file:", xml_files[0])

tree = ET.parse(xml_files[0])
root = tree.getroot()

print("Root tag:", root.tag)

# Print first 30 tags to inspect structure
for elem in list(root.iter())[:30]:
    print(elem.tag)


Example XML file: /home/sreethanu/lidc_ldri_dataset_1/lung_cancer_vl_jepa_lidc/data/raw/LIDC-IDRI/ANNOTATIONS/161-resubmitted-correction-3-9-12.xml
Root tag: {http://www.nih.gov}LidcReadMessage
{http://www.nih.gov}LidcReadMessage
{http://www.nih.gov}ResponseHeader
{http://www.nih.gov}Version
{http://www.nih.gov}MessageId
{http://www.nih.gov}DateRequest
{http://www.nih.gov}TimeRequest
{http://www.nih.gov}RequestingSite
{http://www.nih.gov}ServicingSite
{http://www.nih.gov}TaskDescription
{http://www.nih.gov}CtImageFile
{http://www.nih.gov}SeriesInstanceUid
{http://www.nih.gov}StudyInstanceUID
{http://www.nih.gov}DateService
{http://www.nih.gov}TimeService
{http://www.nih.gov}ResponseDescription
{http://www.nih.gov}ResponseComments
{http://www.nih.gov}readingSession
{http://www.nih.gov}annotationVersion
{http://www.nih.gov}servicingRadiologistID
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}noduleID
{http://www.nih.gov}roi
{http://www.nih.gov}imageZposition
{http://www.nih.

In [14]:
labels_data = []

for series_uid, scores in series_malignancy.items():
    if len(scores) == 0:
        continue

    avg_score = sum(scores) / len(scores)

    # Binary conversion
    label = 1 if avg_score >= 3 else 0

    # Only keep series that exist in processed patches
    if (PATCH_DIR / series_uid).exists():
        labels_data.append({
            "series_uid": series_uid,
            "avg_malignancy": avg_score,
            "label": label
        })

labels_df = pd.DataFrame(labels_data)

print("Final labeled series count:", len(labels_df))
print(labels_df.head())


Final labeled series count: 883
                                          series_uid  avg_malignancy  label
0  1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094...        2.400000      0
1  1.3.6.1.4.1.14519.5.2.1.6279.6001.146429221666...        2.000000      0
2  1.3.6.1.4.1.14519.5.2.1.6279.6001.142154819868...        3.200000      1
3  1.3.6.1.4.1.14519.5.2.1.6279.6001.168833925301...        2.058824      0
4  1.3.6.1.4.1.14519.5.2.1.6279.6001.226889213794...        1.000000      0


In [15]:
labels_df["label"].value_counts()


label
0    475
1    408
Name: count, dtype: int64

In [17]:
metadata_dir = PROJECT_ROOT / "data" / "processed" / "metadata"
metadata_dir.mkdir(parents=True, exist_ok=True)

labels_path = metadata_dir / "labels.csv"
labels_df.to_csv(labels_path, index=False)

print("labels.csv saved at:", labels_path)


labels.csv saved at: /home/sreethanu/lidc_ldri_dataset_1/lung_cancer_vl_jepa_lidc/data/processed/metadata/labels.csv


In [18]:
metadata_path = PROJECT_ROOT / "data" / "processed" / "metadata" / "labels.csv"
metadata = pd.read_csv(metadata_path)


In [20]:
import torch

class_counts = metadata["label"].value_counts().to_dict()
total = sum(class_counts.values())

weights = [
    total / (2 * class_counts[0]),
    total / (2 * class_counts[1])
]

class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print("Corrected class weights:", class_weights)


Corrected class weights: tensor([0.9295, 1.0821], device='cuda:0')


In [21]:
for param in online_encoder.parameters():
    param.requires_grad = False


In [22]:
print(sum(p.requires_grad for p in online_encoder.parameters()))


0


In [23]:
classifier = ClassificationHead(
    embed_dim=config.get("model.encoder.embed_dim"),
    hidden_dims=config.get("model.classifier.hidden_dims"),
    num_classes=2
).to(device)

model = torch.nn.Sequential(
    online_encoder,
    classifier
).to(device)


In [24]:
optimizer = torch.optim.AdamW(
    classifier.parameters(),
    lr=5e-4,
    weight_decay=1e-2
)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)


In [26]:
import pandas as pd
from torch.utils.data import DataLoader, random_split

# Load labels
metadata_path = PROJECT_ROOT / "data" / "processed" / "metadata" / "labels.csv"
metadata = pd.read_csv(metadata_path)

# Build dataset
dataset = LIDCPatchDataset(
    processed_dir=PROJECT_ROOT / "data" / "processed",
    metadata=metadata,
    augment=True
)

# Train/Val split (80/20)
val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.get("data.batch_size"),
    shuffle=True,
    num_workers=config.get("data.num_workers"),
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.get("data.batch_size"),
    shuffle=False,
    num_workers=config.get("data.num_workers"),
    pin_memory=True,
)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))


Loaded 2649 patch samples.
Train size: 2120
Val size: 529


In [27]:
print(len(train_loader))

354


In [38]:
class LIDCClassificationDataset(Dataset):
    def __init__(self, processed_dir, metadata, augment=False):
        self.processed_dir = Path(processed_dir)
        self.metadata = metadata.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        series_uid = row["series_uid"]
        label = row["label"]

        series_path = self.processed_dir / "patches" / series_uid
        patch_files = list(series_path.glob("*.npz"))

        patch_file = patch_files[np.random.randint(len(patch_files))]
        data = np.load(patch_file)

        volume = torch.tensor(data["patch"], dtype=torch.float32)

        # 🔥 Add channel dimension
        if volume.ndim == 3:
            volume = volume.unsqueeze(0)

        return volume, torch.tensor(label, dtype=torch.long)


In [39]:
dataset = LIDCClassificationDataset(
    processed_dir=PROJECT_ROOT / "data" / "processed",
    metadata=metadata,
    augment=True
)


In [40]:
from torch.utils.data import random_split, DataLoader

val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=config.get("data.batch_size"),
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.get("data.batch_size"),
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


In [41]:
epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for volumes, labels in train_loader:
        volumes = volumes.to(device)
        labels = labels.to(device)

        logits = model(volumes)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {total_loss/len(train_loader):.4f}")


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x71c04b113d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1663, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1646, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x71c04b113d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1663, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1646, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoch 1 | Train Loss: 0.7047
Epoch 2 | Train Loss: 0.6998
Epoch 3 | Train Loss: 0.6982
Epoch 4 | Train Loss: 0.6977
Epoch 5 | Train Loss: 0.6956
Epoch 6 | Train Loss: 0.6967
Epoch 7 | Train Loss: 0.6923
Epoch 8 | Train Loss: 0.6927
Epoch 9 | Train Loss: 0.6960
Epoch 10 | Train Loss: 0.6933
Epoch 11 | Train Loss: 0.6964
Epoch 12 | Train Loss: 0.6865
Epoch 13 | Train Loss: 0.6917
Epoch 14 | Train Loss: 0.6920
Epoch 15 | Train Loss: 0.6945
Epoch 16 | Train Loss: 0.6960
Epoch 17 | Train Loss: 0.6888
Epoch 18 | Train Loss: 0.6968
Epoch 19 | Train Loss: 0.6923
Epoch 20 | Train Loss: 0.6886
Epoch 21 | Train Loss: 0.6951
Epoch 22 | Train Loss: 0.6954
Epoch 23 | Train Loss: 0.6935
Epoch 24 | Train Loss: 0.6913
Epoch 25 | Train Loss: 0.6922
Epoch 26 | Train Loss: 0.6940
Epoch 27 | Train Loss: 0.6911
Epoch 28 | Train Loss: 0.6960
Epoch 29 | Train Loss: 0.6920
Epoch 30 | Train Loss: 0.6942


In [42]:
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for volumes, labels in val_loader:

        volumes = volumes.to(device)

        logits = model(volumes)
        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

metrics = compute_classification_metrics(
    np.array(all_labels),
    np.array(all_preds),
    np.array(all_probs)
)

print("Linear Probe Metrics:")
print(metrics)

Linear Probe Metrics:
{'accuracy': 0.4943181818181818, 'balanced_accuracy': 0.4943181818181818, 'precision': 0.47619047619047616, 'recall': 0.11363636363636363, 'f1': 0.1834862385321101, 'sensitivity': np.float64(0.11363636363636363), 'specificity': np.float64(0.875), 'ppv': np.float64(0.47619047619047616), 'npv': np.float64(0.4967741935483871), 'mcc': -0.017527682734987296, 'roc_auc': 0.5860020661157025, 'pr_auc': 0.5466009376697973}


In [43]:
eval_dataset = LIDCClassificationDataset(
    processed_dir=PROJECT_ROOT / "data" / "processed",
    metadata=metadata,
    augment=False
)


In [44]:
model.eval()

PATCH_DIR = PROJECT_ROOT / "data" / "processed" / "patches"

all_scan_preds = []
all_scan_labels = []
all_scan_probs = []

with torch.no_grad():

    for idx in range(len(metadata)):
        row = metadata.iloc[idx]
        series_uid = row["series_uid"]
        true_label = row["label"]

        series_path = PATCH_DIR / series_uid
        patch_files = list(series_path.glob("*.npz"))

        logits_list = []

        for patch_file in patch_files:
            data = np.load(patch_file)
            volume = torch.tensor(data["patch"], dtype=torch.float32)

            # Add channel dimension
            if volume.ndim == 3:
                volume = volume.unsqueeze(0)

            volume = volume.unsqueeze(0).to(device)  # add batch dimension

            logits = model(volume)
            logits_list.append(logits.squeeze(0))

        if len(logits_list) == 0:
            continue

        # 🔥 Average logits across patches
        avg_logits = torch.stack(logits_list).mean(dim=0)

        probs = torch.softmax(avg_logits, dim=0)
        pred = torch.argmax(probs).item()

        all_scan_preds.append(pred)
        all_scan_labels.append(true_label)
        all_scan_probs.append(probs[1].item())


In [45]:
metrics = compute_classification_metrics(
    np.array(all_scan_labels),
    np.array(all_scan_preds),
    np.array(all_scan_probs)
)

print("Scan-Level Aggregated Metrics:")
print(metrics)


Scan-Level Aggregated Metrics:
{'accuracy': 0.5560588901472253, 'balanced_accuracy': 0.5323993808049536, 'precision': 0.5487804878048781, 'recall': 0.22058823529411764, 'f1': 0.3146853146853147, 'sensitivity': np.float64(0.22058823529411764), 'specificity': np.float64(0.8442105263157895), 'ppv': np.float64(0.5487804878048781), 'npv': np.float64(0.5577190542420027), 'mcc': 0.08307248904818075, 'roc_auc': 0.5564654282765737, 'pr_auc': 0.5299054514978057}
